# Notebook 1: Hand-Coded 1D Poisson Solver

## Objective
Implement a finite element solver for the 1D Poisson problem **from scratch** using numpy.

**Problem:** 
$$-\frac{d^2u}{dx^2} = f(x) \quad \text{in } (0, 1)$$
$$u(0) = 0, \quad u(1) = 0$$

We'll use **piecewise linear basis functions** (P1 elements) and **assembly**.

## Why Hand-Code?
This is the **single most important step** in learning FEM. You'll see exactly how assembly works, why the stiffness matrix has the form it does, and how BCs are enforced. Next week, FEniCS will do all this for you—but you'll understand what's happening under the hood.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

---
## Step 1: Define the Problem

Let's solve with a simple manufactured source:
$$f(x) = -6x$$

**Exact solution:** $u(x) = x(1 - x)$ (verify by substitution).

This is a great test case because we know the exact answer.

In [ ]:
# Problem parameters
n_elements = 10  # Number of elements (start small)
n_nodes = n_elements + 1  # P1: one node per element vertex
x = np.linspace(0, 1, n_nodes)  # Node coordinates
h = x[1] - x[0]  # Element size (uniform mesh)

print(f"Mesh: {n_nodes} nodes, {n_elements} elements")
print(f"Element size: h = {h}")

# Source function and exact solution
def f_source(x):
    """Load vector: -6*x"""
    return -6 * x

def u_exact(x):
    """Exact solution: x*(1-x)"""
    return x * (1 - x)

print(f"\nExact solution at midpoints:")
for xi in [0.25, 0.5, 0.75]:
    print(f"  u({xi}) = {u_exact(xi):.4f}")

---
## Step 2: Assemble the Global Stiffness Matrix

For a uniform 1D mesh with P1 elements, the local stiffness matrix for element $e$ is:
$$K^e = \frac{1}{h} \begin{pmatrix} 1 & -1 \\ -1 & 1 \end{pmatrix}$$

and the local load vector is:
$$f^e_i = \int_{x_k}^{x_{k+1}} f(x) \phi_i(x) \, dx$$

We'll use **1-point Gauss quadrature** at the element center for simplicity (exact for linear f on P1).

In [ ]:
# Initialize global matrix and RHS
K = np.zeros((n_nodes, n_nodes))
f = np.zeros(n_nodes)

# Local stiffness matrix for P1 element
K_local = (1.0 / h) * np.array([[1.0, -1.0],
                                  [-1.0, 1.0]])

print("Local stiffness matrix (P1):")
print(K_local)
print()

# Assembly loop
for e in range(n_elements):
    # Global node numbers for this element
    nodes_local = [e, e + 1]  # Left and right nodes
    
    # Add local stiffness to global
    for i in range(2):
        for j in range(2):
            K[nodes_local[i], nodes_local[j]] += K_local[i, j]
    
    # Load vector: 1-point Gauss quadrature at element center
    x_center = (x[e] + x[e + 1]) / 2.0
    f_center = f_source(x_center)
    
    # For P1, the basis functions at the center are [0.5, 0.5]
    # So f^e = [f_center * h / 2, f_center * h / 2]
    f[nodes_local[0]] += f_center * h / 2.0
    f[nodes_local[1]] += f_center * h / 2.0

print("Global stiffness matrix (first 5x5 block):")
print(K[:5, :5])
print()
print("Load vector:")
print(f)

---
## Step 3: Enforce Dirichlet Boundary Conditions

We have $u(0) = 0$ and $u(1) = 0$. These are **essential** (Dirichlet) BCs.

To enforce them, we modify the linear system:
- Set row 0 (node at $x=0$): $K[0, :] \to [1, 0, 0, \ldots]$, $f[0] \to 0$
- Set row $n$ (node at $x=1$): $K[n, :] \to [0, \ldots, 1]$, $f[n] \to 0$

In [ ]:
# Enforce u(0) = 0 (node 0)
K[0, :] = 0.0
K[0, 0] = 1.0
f[0] = 0.0

# Enforce u(1) = 0 (last node)
K[-1, :] = 0.0
K[-1, -1] = 1.0
f[-1] = 0.0

print("Stiffness matrix after BCs (first 5x5 block):")
print(K[:5, :5])
print()
print("Load vector after BCs:")
print(f)

---
## Step 4: Solve the Linear System

Now we solve $K \mathbf{u} = \mathbf{f}$ for the unknown nodal values.

In [ ]:
# Solve
u_fem = solve(K, f)

print("FEM solution (first 10 nodes):")
print(u_fem[:10])
print()

# Exact solution at nodes
u_exact_vals = u_exact(x)

print("Exact solution (first 10 nodes):")
print(u_exact_vals[:10])
print()

# Error at nodes
error = u_fem - u_exact_vals
l2_error = np.sqrt(np.trapz(error**2, x))
max_error = np.max(np.abs(error))

print(f"L2 error: {l2_error:.2e}")
print(f"Max error: {max_error:.2e}")

---
## Step 5: Visualization & Validation

Plot the FEM solution against the exact solution and check convergence.

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Solution comparison
ax = axes[0]
x_fine = np.linspace(0, 1, 200)
u_exact_fine = u_exact(x_fine)

ax.plot(x_fine, u_exact_fine, 'b-', linewidth=2, label='Exact')
ax.plot(x, u_fem, 'ro', markersize=5, label=f'FEM (n={n_elements})')
ax.set_xlabel('x')
ax.set_ylabel('u(x)')
ax.set_title('1D Poisson: Solution')
ax.legend()
ax.grid(True, alpha=0.3)

# Error
ax = axes[1]
ax.plot(x, error, 'r-', linewidth=1, marker='o', markersize=4)
ax.set_xlabel('x')
ax.set_ylabel('u_exact - u_fem')
ax.set_title('Pointwise Error')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/poisson_solution.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved.")

---
## Step 6: Convergence Study

Now let's refine the mesh and check that the error decreases as predicted by theory.

For P1 elements: $\|u - u_h\|_{L^2} = O(h^2)$

On a log-log plot, the slope should be 2.

In [ ]:
# Convergence study
n_elements_list = np.array([4, 8, 16, 32, 64, 128])
l2_errors = []
h_values = []

for n_elem in n_elements_list:
    n_node = n_elem + 1
    x_mesh = np.linspace(0, 1, n_node)
    h_mesh = x_mesh[1] - x_mesh[0]
    h_values.append(h_mesh)
    
    # Assemble and solve (same as above, packaged)
    K_mesh = np.zeros((n_node, n_node))
    f_mesh = np.zeros(n_node)
    K_local_mesh = (1.0 / h_mesh) * np.array([[1.0, -1.0], [-1.0, 1.0]])
    
    for e in range(n_elem):
        nodes = [e, e + 1]
        for i in range(2):
            for j in range(2):
                K_mesh[nodes[i], nodes[j]] += K_local_mesh[i, j]
        
        x_c = (x_mesh[e] + x_mesh[e + 1]) / 2.0
        f_mesh[nodes[0]] += f_source(x_c) * h_mesh / 2.0
        f_mesh[nodes[1]] += f_source(x_c) * h_mesh / 2.0
    
    # BCs
    K_mesh[0, :] = 0.0
    K_mesh[0, 0] = 1.0
    f_mesh[0] = 0.0
    K_mesh[-1, :] = 0.0
    K_mesh[-1, -1] = 1.0
    f_mesh[-1] = 0.0
    
    # Solve
    u_sol = solve(K_mesh, f_mesh)
    u_ex = u_exact(x_mesh)
    
    # Error
    err = np.sqrt(np.trapz((u_sol - u_ex)**2, x_mesh))
    l2_errors.append(err)

h_values = np.array(h_values)
l2_errors = np.array(l2_errors)

# Print table
print("Convergence Table:")
print("-" * 60)
print(f"{'n_elem':>10} | {'h':>12} | {'L2 error':>12} | {'Rate':>8}")
print("-" * 60)

for i, n in enumerate(n_elements_list):
    if i == 0:
        rate = "-"
    else:
        rate = np.log(l2_errors[i-1] / l2_errors[i]) / np.log(h_values[i-1] / h_values[i])
        rate = f"{rate:.2f}"
    print(f"{n:>10} | {h_values[i]:>12.4e} | {l2_errors[i]:>12.4e} | {rate:>8}")

print("-" * 60)
print("Expected rate for P1: 2.0")

In [ ]:
# Plot convergence on log-log
fig, ax = plt.subplots(figsize=(8, 6))

ax.loglog(h_values, l2_errors, 'ro-', linewidth=2, markersize=8, label='FEM (P1)')

# Reference line: slope = 2 (h^2 convergence)
h_ref = np.array([h_values[0], h_values[-1]])
err_ref = l2_errors[0] * (h_ref / h_values[0])**2
ax.loglog(h_ref, err_ref, 'k--', linewidth=1, alpha=0.5, label='h^2 (reference)')

ax.set_xlabel('Mesh size h', fontsize=12)
ax.set_ylabel('L2 error', fontsize=12)
ax.set_title('Convergence: 1D Poisson with P1 Elements', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/convergence.png', dpi=150, bbox_inches='tight')
plt.show()

print("Convergence plot saved.")

---
## Summary

You've just implemented a complete FEM solver from scratch! Key takeaways:

1. **Weak formulation** reduces regularity requirements on the solution.
2. **Assembly** combines local element contributions into a global matrix.
3. **Boundary conditions** are enforced by modifying rows of the system.
4. **Convergence** follows the theory: $\|u - u_h\|_{L^2} = O(h^2)$ for P1.

## Next Week
FEniCS will do all of this automatically. But now you understand what's happening under the hood.

**Exercise:** Try changing the source function $f(x)$ or the BCs. What happens to the solution? Does convergence still hold?